# 1. Project Introduction

**Title:** Learning Hidden Recipe Preferences Using Restricted Boltzmann Machines (RBM)

A recipe platform has unlabeled combinations of ingredients and user choices and wants to learn hidden preference features. In this notebook we implement a simple **Restricted Boltzmann Machine (RBM)** using `scikit-learn`'s `BernoulliRBM`, analyze its hidden-unit activations, compare representations across different user preferences, and interpret the latent preference patterns it discovers -- all without ever telling the model what those patterns should be.

# 2. Import Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neural_network import BernoulliRBM
from sklearn.decomposition import PCA

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 110

# 3. Load Dataset

In [ ]:
DATASET_PATH = '../dataset/recipe_data.csv'
recipe_dataframe = pd.read_csv(DATASET_PATH)
recipe_dataframe.head()

# 4. Explore Dataset

In [ ]:
print('Dataset shape:', recipe_dataframe.shape)
print('\nColumn names:', list(recipe_dataframe.columns))
print('\nMissing values per column:')
print(recipe_dataframe.isnull().sum())
recipe_dataframe.info()

# 5. Data Preprocessing

We select the 17 binary ingredient/characteristic columns to form the visible-unit feature matrix. Since `BernoulliRBM` models visible units as Bernoulli (0/1) variables, our already-binary data requires no additional scaling or encoding.

In [ ]:
FEATURE_COLUMNS = [
    'Vegetarian', 'Chicken', 'Egg', 'Cheese', 'Milk', 'Paneer', 'Rice',
    'Vegetables', 'Fruits', 'Sugar', 'Chocolate', 'Spices', 'Chili',
    'Protein', 'Healthy', 'Sweet', 'Spicy'
]

feature_matrix = recipe_dataframe[FEATURE_COLUMNS].values.astype(np.float32)
print('Feature matrix shape (samples x visible units):', feature_matrix.shape)
print('Unique values present:', np.unique(feature_matrix))

# 6. RBM Model Creation

In [ ]:
NUM_HIDDEN_UNITS = 6
RANDOM_STATE = 42

rbm_model = BernoulliRBM(
    n_components=NUM_HIDDEN_UNITS,   # number of hidden units to learn
    learning_rate=0.05,              # step size for weight updates
    batch_size=10,                   # mini-batch size during training
    n_iter=200,                      # number of training epochs
    random_state=RANDOM_STATE,
)
rbm_model

# 7. Model Training

In [ ]:
rbm_model.fit(feature_matrix)
print('Training complete.')
print('Visible units:', feature_matrix.shape[1])
print('Hidden units:', NUM_HIDDEN_UNITS)

# 8. Hidden Unit Activations

In [ ]:
hidden_activations = rbm_model.transform(feature_matrix)
hidden_columns = [f'Hidden_Unit_{i+1}' for i in range(NUM_HIDDEN_UNITS)]

activation_table = pd.DataFrame(hidden_activations, columns=hidden_columns)
activation_table.insert(0, 'Recipe_Name', recipe_dataframe['Recipe_Name'].values)
activation_table.insert(0, 'Recipe_ID', recipe_dataframe['Recipe_ID'].values)
activation_table.insert(0, 'User_ID', recipe_dataframe['User_ID'].values)
activation_table.head(10)

# 9. Latent Pattern Analysis

We inspect `rbm_model.components_` (the learned weight matrix) to see which visible features each hidden unit is most strongly associated with. This interpretation happens *after* training -- the RBM was never told what these hidden units should represent.

In [ ]:
weight_matrix = rbm_model.components_
TOP_N = 4

for hidden_index in range(weight_matrix.shape[0]):
    weights_for_unit = weight_matrix[hidden_index]
    top_indices = np.argsort(weights_for_unit)[::-1][:TOP_N]
    top_features = [(FEATURE_COLUMNS[i], round(float(weights_for_unit[i]), 3)) for i in top_indices]
    print(f'Hidden Unit {hidden_index + 1}: {top_features}')

# 10. Visualization

In [ ]:
# Hidden-unit activation heatmap (sample of 30 records)
fig, ax = plt.subplots(figsize=(8, 10))
sns.heatmap(activation_table[hidden_columns].head(30), cmap='viridis', ax=ax,
            cbar_kws={'label': 'Activation Strength'})
ax.set_title('Hidden Unit Activations (Sample of 30 Records)')
plt.show()

In [ ]:
# Latent preference pattern heatmap: hidden units vs visible features
fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(weight_matrix, cmap='coolwarm', center=0,
            xticklabels=FEATURE_COLUMNS,
            yticklabels=[f'Hidden {i+1}' for i in range(weight_matrix.shape[0])],
            cbar_kws={'label': 'Weight Strength'}, ax=ax)
ax.set_title('Latent Preference Patterns: Hidden Units vs Ingredient Features')
plt.xticks(rotation=45, ha='right')
plt.show()

In [ ]:
# PCA projection of hidden representations to visualize preference clusters
pca_model = PCA(n_components=2, random_state=RANDOM_STATE)
pca_coordinates = pca_model.fit_transform(activation_table[hidden_columns].values)

fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(pca_coordinates[:, 0], pca_coordinates[:, 1],
                      c=recipe_dataframe['Vegetarian'], cmap='coolwarm',
                      alpha=0.75, edgecolor='k', linewidth=0.3)
handles, _ = scatter.legend_elements()
ax.legend(handles, ['Non-Vegetarian', 'Vegetarian'], title='Recipe Type')
ax.set_title('Preference Clusters (PCA of Hidden Representations)')
ax.set_xlabel('Principal Component 1')
ax.set_ylabel('Principal Component 2')
plt.show()

# 11. Comparison of User Preferences

In [ ]:
group_definitions = {
    'Vegetarian': recipe_dataframe['Vegetarian'],
    'Spicy': recipe_dataframe['Spicy'],
    'Sweet': recipe_dataframe['Sweet'],
    'Healthy': recipe_dataframe['Healthy'],
}

comparison_results = {}
for characteristic_name, characteristic_series in group_definitions.items():
    group_yes_mean = activation_table[hidden_columns][characteristic_series == 1].mean()
    group_no_mean = activation_table[hidden_columns][characteristic_series == 0].mean()
    comparison_results[f'{characteristic_name}=1'] = group_yes_mean
    comparison_results[f'{characteristic_name}=0'] = group_no_mean

comparison_df = pd.DataFrame(comparison_results).T
comparison_df

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
comparison_df.T.plot(kind='bar', ax=ax, colormap='tab10')
ax.set_title('Average Hidden-Unit Activation by User Preference Group')
ax.set_xlabel('Hidden Units')
ax.set_ylabel('Average Activation')
ax.legend(title='Preference Group', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

# 12. Conclusion

The RBM successfully learned hidden units that align with intuitive food-preference
concepts (sweet, spicy, vegetarian/healthy) purely from unlabeled binary
ingredient data. Comparing average hidden activations across known groups
(e.g. Spicy = 1 vs Spicy = 0) confirms the latent space captures real,
interpretable structure. This demonstrates the power of unsupervised feature
learning for recommendation-style problems, and this approach could be
extended with a larger real-world dataset, more hidden units, or combined
with a downstream recommender system.